# Data Understanding 1: `application_train`

The main table of the Home Credit Default Risk dataset: one loan application per row, with everything known about the applicant at the moment they applied, plus the target.

**TARGET** = 1 if the applicant later had payment difficulties, 0 otherwise. This is what the PD model will predict.

Each table is profiled with the same five questions:

1. **Grain:** what does one row represent?
2. **Real rows:** what does the data actually look like?
3. **Types:** what type is each column, and what does it really mean?
4. **Missing data:** where are the blanks, and why?
5. **Ranges:** do the values make sense, and what looks wrong?

## Setup

DuckDB queries the CSV files directly, so no table is loaded fully into memory. Only small, aggregated results are passed to pandas for display. Run the cells top to bottom.

In [13]:
import duckdb
import pandas as pd

APP = 'data/raw/application_train.csv'

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)

---
## Q1. Grain

If total rows equal the number of unique `SK_ID_CURR`, every row is a different applicant.

In [14]:
duckdb.sql(f'''
    SELECT
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT SK_ID_CURR) AS unique_applicants
    FROM '{APP}'
''').show()

┌────────────┬───────────────────┐
│ total_rows │ unique_applicants │
│   int64    │       int64       │
├────────────┼───────────────────┤
│     307511 │            307511 │
└────────────┴───────────────────┘



**Target balance:** the share of applicants who defaulted.

In [15]:
duckdb.sql(f'''
    SELECT
        TARGET,
        COUNT(*) AS applicants,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM '{APP}'
    GROUP BY TARGET
    ORDER BY TARGET
''').show()

┌────────┬────────────┬────────┐
│ TARGET │ applicants │  pct   │
│ int64  │   int64    │ double │
├────────┼────────────┼────────┤
│      0 │     282686 │  91.93 │
│      1 │      24825 │   8.07 │
└────────┴────────────┴────────┘



**Findings**

- 307,511 rows and 307,511 unique `SK_ID_CURR`: **one row = one applicant**, no duplicates.
- 24,825 applicants defaulted: a **default rate of 8.07%**.
- This is the central table. Every other table has many rows per applicant and must be aggregated to one row per `SK_ID_CURR` before it is joined here.
- With only 8% defaults, accuracy is a misleading metric: predicting "no default" for everyone scores 92%. Models will be judged by AUC / Gini instead.

---
## Q2. Real rows

122 columns are too wide to read, so a 3-row sample is transposed: columns become rows.

In [16]:
app_sample = duckdb.sql(f"SELECT * FROM '{APP}' LIMIT 3").df()
app_sample.T

,0,1,2
SK_ID_CURR,100002,100003,100004
TARGET,1,0,0
NAME_CONTRACT_TYPE,Cash loans,Cash loans,Revolving loans
CODE_GENDER,M,F,M
FLAG_OWN_CAR,N,N,Y
FLAG_OWN_REALTY,Y,N,Y
CNT_CHILDREN,0,0,0
AMT_INCOME_TOTAL,202500.0,270000.0,67500.0
AMT_CREDIT,406597.5,1293502.5,135000.0
AMT_ANNUITY,24700.5,35698.5,6750.0


The data dictionary explains columns whose names are unclear.

In [17]:
desc = pd.read_csv('data/raw/HomeCredit_columns_description.csv', encoding='latin-1')
desc[desc['Table'].str.startswith('application') & desc['Row'].str.contains('EXT_SOURCE|FLAG_DOC|DAYS_')][['Row', 'Description']]

,Row,Description
17,DAYS_BIRTH,Client's age in days at the time of application
18,DAYS_EMPLOYED,How many days before the application the person started current employment
19,DAYS_REGISTRATION,How many days before the application did client change his registration
20,DAYS_ID_PUBLISH,How many days before the application did client change the identity document with which he applied for the loan
41,EXT_SOURCE_1,Normalized score from external data source
42,EXT_SOURCE_2,Normalized score from external data source
43,EXT_SOURCE_3,Normalized score from external data source
95,DAYS_LAST_PHONE_CHANGE,How many days before application did client change phone
96,FLAG_DOCUMENT_2,Did client provide document 2
97,FLAG_DOCUMENT_3,Did client provide document 3


**Findings: column families**

| Family | Examples | Notes |
|---|---|---|
| Money | `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_GOODS_PRICE` | Currency not disclosed: treat as units |
| Personal and family | `CODE_GENDER`, `NAME_FAMILY_STATUS`, `CNT_CHILDREN` | |
| Work and income | `NAME_INCOME_TYPE`, `OCCUPATION_TYPE`, `ORGANIZATION_TYPE` | |
| Time | `DAYS_BIRTH`, `DAYS_EMPLOYED`, `DAYS_REGISTRATION` | Negative days before the application |
| External scores | `EXT_SOURCE_1/2/3` | Scaled 0 to 1, source undisclosed |
| Documents | `FLAG_DOCUMENT_2` to `_21` | 1 = document N submitted |
| Building stats | `..._AVG`, `..._MODE`, `..._MEDI` | Normalised housing information |
| Credit enquiries | `AMT_REQ_CREDIT_BUREAU_...` | Recent credit checks |

Column names say what a column is meant to hold; real rows show what it actually holds (negative day counts, Y/N text, blanks).

---
## Q3. Column types and meaning

In [18]:
duckdb.sql(f'''
    SELECT column_type, COUNT(*) AS num_columns
    FROM (DESCRIBE SELECT * FROM '{APP}')
    GROUP BY column_type
    ORDER BY num_columns DESC
''').show()

┌─────────────┬─────────────┐
│ column_type │ num_columns │
│   varchar   │    int64    │
├─────────────┼─────────────┤
│ DOUBLE      │          65 │
│ BIGINT      │          41 │
│ VARCHAR     │          15 │
│ BOOLEAN     │           1 │
└─────────────┴─────────────┘



The text columns:

In [19]:
duckdb.sql(f'''
    SELECT column_name
    FROM (DESCRIBE SELECT * FROM '{APP}')
    WHERE column_type = 'VARCHAR'
''').show()

┌────────────────────────────┐
│        column_name         │
│          varchar           │
├────────────────────────────┤
│ NAME_CONTRACT_TYPE         │
│ CODE_GENDER                │
│ FLAG_OWN_CAR               │
│ FLAG_OWN_REALTY            │
│ NAME_TYPE_SUITE            │
│ NAME_INCOME_TYPE           │
│ NAME_EDUCATION_TYPE        │
│ NAME_FAMILY_STATUS         │
│ NAME_HOUSING_TYPE          │
│ OCCUPATION_TYPE            │
│ WEEKDAY_APPR_PROCESS_START │
│ ORGANIZATION_TYPE          │
│ FONDKAPREMONT_MODE         │
│ HOUSETYPE_MODE             │
│ WALLSMATERIAL_MODE         │
└────────────────────────────┘
           15 rows          



**Findings**

- The 15 text columns are all **categories** (contract type, gender, education, family status, housing, occupation, organisation...). None are measurements. They need encoding before modelling.
- **Storage is not meaning.** Yes/no information is stored three different ways in one table:
  - 0/1 integers: `FLAG_DOCUMENT_*`, `FLAG_MOBIL`, `FLAG_EMAIL`, `TARGET`
  - 'Y'/'N' text: `FLAG_OWN_CAR`, `FLAG_OWN_REALTY`
  - a boolean: `EMERGENCYSTATE_MODE`
- A 0/1 flag can be averaged to get a share (`AVG(TARGET)` is the default rate), but it is still a category, not a quantity.

---
## Q4. Missing data

In [20]:
app_summary = duckdb.sql(f"SUMMARIZE SELECT * FROM '{APP}'").df()
app_summary[['column_name', 'column_type', 'null_percentage']].sort_values('null_percentage', ascending=False).head(50)

,column_name,column_type,null_percentage
48,COMMONAREA_AVG,DOUBLE,69.87
62,COMMONAREA_MODE,DOUBLE,69.87
76,COMMONAREA_MEDI,DOUBLE,69.87
84,NONLIVINGAPARTMENTS_MEDI,DOUBLE,69.43
70,NONLIVINGAPARTMENTS_MODE,DOUBLE,69.43
56,NONLIVINGAPARTMENTS_AVG,DOUBLE,69.43
86,FONDKAPREMONT_MODE,VARCHAR,68.39
54,LIVINGAPARTMENTS_AVG,DOUBLE,68.35
82,LIVINGAPARTMENTS_MEDI,DOUBLE,68.35
68,LIVINGAPARTMENTS_MODE,DOUBLE,68.35


In [21]:
print('Columns more than 50% empty:',
      (app_summary['null_percentage'].astype(float) > 50).sum(), 'of', len(app_summary))

Columns more than 50% empty: 41 of 122


**Testing a theory:** is `OWN_CAR_AGE` blank because the applicant has no car? If so, the blanks should sit almost entirely with `FLAG_OWN_CAR = 'N'`.

In [22]:
duckdb.sql(f'''
    SELECT
        FLAG_OWN_CAR,
        COUNT(*) AS applicants,
        SUM(CASE WHEN OWN_CAR_AGE IS NULL THEN 1 ELSE 0 END) AS car_age_blank
    FROM '{APP}'
    GROUP BY FLAG_OWN_CAR
''').show()

┌──────────────┬────────────┬───────────────┐
│ FLAG_OWN_CAR │ applicants │ car_age_blank │
│   varchar    │   int64    │    int128     │
├──────────────┼────────────┼───────────────┤
│ N            │     202924 │        202924 │
│ Y            │     104587 │             5 │
└──────────────┴────────────┴───────────────┘



**Findings**

- About **a third of the 122 columns are more than half empty**, mostly the building statistics block (`_AVG`, `_MODE`, `_MEDI`).
- `EXT_SOURCE_1`, one of the strongest predictors, is also more than half empty.
- **Two kinds of missing data:**
  - *Doesn't apply:* the question has no answer for this person, so the blank carries meaning.
  - *Truly unknown:* a value exists but was not recorded.
- `OWN_CAR_AGE`: only **5 car owners** have a blank car age. Blank means **no car** (doesn't apply). Filling it with an average would invent cars for people who have none.

---
## Q5. Ranges and anomalies

`DAYS_` columns count days before the application, so they should be negative.

In [23]:
app_summary[app_summary['column_name'].str.startswith('DAYS')][['column_name', 'min', 'max', 'avg']]

,column_name,min,max,avg
17,DAYS_BIRTH,-25229,-7489,-16036.995066843137
18,DAYS_EMPLOYED,-17912,365243,63815.04590404896
19,DAYS_REGISTRATION,-24672.0,0.0,-4986.120327538419
20,DAYS_ID_PUBLISH,-7197,0,-2994.2023732484367
95,DAYS_LAST_PHONE_CHANGE,-4292.0,0.0,-962.8587883320868


`DAYS_EMPLOYED` has a positive maximum. Who has it?

In [24]:
duckdb.sql(f'''
    SELECT
        NAME_INCOME_TYPE,
        COUNT(*) AS applicants
    FROM '{APP}'
    WHERE DAYS_EMPLOYED = 365243
    GROUP BY NAME_INCOME_TYPE
    ORDER BY applicants DESC
''').show()

┌──────────────────┬────────────┐
│ NAME_INCOME_TYPE │ applicants │
│     varchar      │   int64    │
├──────────────────┼────────────┤
│ Pensioner        │      55352 │
│ Unemployed       │         22 │
└──────────────────┴────────────┘



**Findings**

- `DAYS_BIRTH`: applicants are about **20 to 69 years old**. Sensible.
- `DAYS_EMPLOYED` max = **365,243**: positive and about 1,000 years. A **placeholder**, not a real value.
- It belongs almost entirely to **pensioners**, plus some **unemployed** applicants: people with no current job.
- Treatment for modelling: replace 365243 with NULL and add a "not employed" flag, so the information is kept without the fake number distorting averages.
- An **error** is impossible (a 1,000-year job); an **outlier** is rare but possible (49 years in one job). Errors are neutralised; outliers are usually capped.

---
## Summary

| Question | `application_train` |
|---|---|
| Grain | One row per applicant (307,511), default rate 8.07% |
| Real rows | 122 columns in clear families; dictionary needed for `EXT_SOURCE` and `FLAG_DOCUMENT` |
| Types | 15 categorical text columns; yes/no stored three ways |
| Missing | ~1/3 of columns over 50% empty (building stats); `OWN_CAR_AGE` blank = no car |
| Anomalies | `DAYS_EMPLOYED = 365243` placeholder for pensioners and unemployed |

**Carried into feature engineering:** encode categoricals, treat `365243` as NULL plus a flag, keep "doesn't apply" blanks meaningful rather than filling them.